In [3]:
import zipfile, gc, os
import numpy as np
from modules import (
    MusicDataLoaderPl,
    NonePersonalizedRecommender,
    ContentBasedRecommender,
    CollaborativeFilteringRecommender
    )

# connecting google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Loading the dataset

I'm using a custom Million Song Dataset: https://yadi.sk/d/AJ2yfSnhcFTLjQ?w=1
And since I'm not running it locally, I uploaded the dataset on my google drive, home_path is the path to the dataset wheteer it's in the drive or in local

In [4]:
home_path = "drive/MyDrive/ds3/src/datasets/"
if len(os.listdir(home_path)) == 0:
  with zipfile.ZipFile(home_path+"p02_mxm_dataset_train.txt.zip") as zip_ref:
      zip_ref.extractall(home_path)
  with zipfile.ZipFile(home_path+"p02_train_triplets.txt.zip") as zip_ref:
      zip_ref.extractall(home_path)
os.listdir(home_path)

['p02_msd_tagtraum_cd2.cls',
 'p02_mxm_dataset_train.txt.zip',
 'p02_train_triplets.txt.zip',
 'p02_unique_tracks.txt',
 'extr',
 'mxm_dataset_train.txt',
 'train_triplets.txt',
 '__MACOSX']

Datasets to use:<br>
- p02_msd_tagtraum_cd2.cls<br>
- p02_unique_tracks.txt<br>
- train_triplets.txt
<h3>We are not using musiXmatch dataset! Bacuse there's something wrong with it and it's always filling up our non-existent precious RAM!</h3>

In [5]:
loader = MusicDataLoaderPl(
    triplets_path=home_path+"train_triplets.txt",
    unique_tracks_path=home_path+"p02_unique_tracks.txt",
    genres_path=home_path+"p02_msd_tagtraum_cd2.cls",
    sample_fraction=0.5)
df, tracks_df, triplets_df, genres_df = loader.load_data()

1. Loading unique tracks dataset...
2. Loading triplets dataset...
   Sampling 50.0% of triplets dataset...
3. Loading genres dataset...
4. Merging datasets...


In [6]:
print(df.shape)
df.head(5)

(24831952, 8)


user_id,song_id,play_count,track_id,artist,title,majority-genre,minorillty-genre
cat,cat,i32,cat,str,str,str,str
"""ac3f9932eb88f33c7aa78bbba45183…","""SOVGSEL12A8C141591""",1,"""TRBJRQN128F4298744""","""Lifehouse""","""Blind""","""Rock""",null
"""cc3026269e817841fa70a63d7895f0…","""SOWMGYB12A8C136507""",1,"""TRNCAZY128F4272748""","""Christopher O'Riley""","""Black Star""","""Rock""","""Jazz"""
"""db226f9e76aa906f3dc8dbebcce326…","""SOCDMKB12A8C1344F5""",1,"""TRUOISO128F425623D""","""Mor W.A.""","""Co Nas Motywuje (Remix)""",null,null
"""89cf0e4c3b3d6a140f1427b946096e…","""SOOGWEB12A67020B58""",1,"""TRPLMXP128E0792700""","""Rob Zombie""","""House Of 1000 Corpses""",null,null
"""e52dc967e2ae87165f5850418ce89b…","""SODRHMF12AB0189504""",1,"""TRIHDCP12903CB30B4""","""Saeed Younan""","""Backroom Honey (Original Mix)""",null,null


# Modeling


## Top 250 Tracks | Top 100 tracks by genre

In [7]:
npr = NonePersonalizedRecommender(df)
genre = "Metal"

print("Top 250 songs:")
npr.get_top_250()

Top 250 songs:


index_number,artist,title,play_count
u32,str,str,i32
1,"""Dwight Yoakam""","""You're The One""",362737
2,"""Björk""","""Undo""",326957
3,"""Kings Of Leon""","""Revelry""",263405
4,"""Harmonia""","""Sehr kosmisch""",213049
5,"""Barry Tuckwell/Academy of St M…","""Horn Concerto No. 4 in E flat …",195926
…,…,…,…
246,"""Lady GaGa""","""Monster""",18685
247,"""Kings Of Leon""","""Be Somebody""",18674
248,"""Metallica / Marianne Faithfull""","""The Memory Remains""",18537


In [8]:
genre = "Jazz"
print(f'Top 100 tracks by "{genre}" genre:')
print(npr.get_top_100_by_genre(genre))

Top 100 tracks by "Jazz" genre:
shape: (100, 4)
┌──────────────┬───────────────────────────┬────────────────────────────┬────────────┐
│ index_number ┆ artist                    ┆ title                      ┆ play_count │
│ ---          ┆ ---                       ┆ ---                        ┆ ---        │
│ u32          ┆ str                       ┆ str                        ┆ i32        │
╞══════════════╪═══════════════════════════╪════════════════════════════╪════════════╡
│ 1            ┆ Ron Carter                ┆ I CAN'T GET STARTED        ┆ 80820      │
│ 2            ┆ Jacky Terrasson           ┆ Le Jardin d'Hiver          ┆ 37264      │
│ 3            ┆ Erin Bode                 ┆ Here_ There And Everywhere ┆ 22697      │
│ 4            ┆ Michael Bublé             ┆ Home (Album Version)       ┆ 13509      │
│ 5            ┆ Bohren & Der Club Of Gore ┆ Welk                       ┆ 12162      │
│ …            ┆ …                         ┆ …                          ┆ …       

## Collections

In [9]:
keywords = ["love", "war"]
for keyword in keywords:
  print(f'50 Most played songs with "{keyword}" in the title:')
  cbr = ContentBasedRecommender(tracks_df, triplets_df)
  print(cbr.get_collection(keyword))

50 Most played songs with "love" in the title:
shape: (50, 4)
┌──────────────┬────────────────────────┬─────────────────────────────────┬────────────┐
│ index_number ┆ artist                 ┆ title                           ┆ play_count │
│ ---          ┆ ---                    ┆ ---                             ┆ ---        │
│ u32          ┆ str                    ┆ str                             ┆ i32        │
╞══════════════╪════════════════════════╪═════════════════════════════════╪════════════╡
│ 1            ┆ Bill Withers           ┆ Make Love To Your Mind          ┆ 72122      │
│ 2            ┆ Justin Bieber          ┆ Somebody To Love                ┆ 55448      │
│ 3            ┆ Justin Bieber          ┆ Somebody To Love                ┆ 55448      │
│ 4            ┆ Florence + The Machine ┆ Cosmic Love                     ┆ 47677      │
│ 5            ┆ Taylor Swift           ┆ Love Story                      ┆ 44164      │
│ …            ┆ …                      ┆ …     

## Collaborative Recommendation System

In [10]:
cfr = CollaborativeFilteringRecommender(df)
cfr.prepare_matrices(use_vectorizer=False)

# random user and song ids for recommending songs
user_id = df['user_id'][np.random.randint(0, len(df))]
song_id = df['song_id'][np.random.randint(0, len(df))]

# getting the recommendations
for_user_ids, for_user = cfr.recommend_for_user(user_id)
for_song_ids, for_song = cfr.recommend_for_song(song_id)

print("Calculating the p@k...")
sample_size, k = 100, 10
for_user_pk = cfr.evaluate_user_model(sample_size=sample_size, k=k)
for_song_pk = cfr.evaluate_song_model(sample_size=sample_size, k=k)

print("\nPeople similar to you listening:")
print(for_user)
print(f"Average Precision@{k} (p@k): {for_user_pk*100:.2f}%")
print("\nPeople who listen to this track usually listen:")
print(for_song)
print(f"Average Precision@{k} (p@k): {for_song_pk*100:.2f}%")

print("\n---- END RESULTS ------------------------")
print(f"Users similarity recommendation system Average Precision@{k} (p@k): {for_user_pk*100:.2f}%")
print(f"Tracks similarity recommendation system Average Precision@{k} (p@k): {for_song_pk*100:.2f}%")

Calculating the p@k...

People similar to you listening:
shape: (10, 3)
┌──────────────┬────────────────────────┬─────────────────────────────────┐
│ index_number ┆ artist                 ┆ title                           │
│ ---          ┆ ---                    ┆ ---                             │
│ u32          ┆ str                    ┆ str                             │
╞══════════════╪════════════════════════╪═════════════════════════════════╡
│ 1            ┆ Linkin Park            ┆ Crawling (Album Version)        │
│ 2            ┆ Muse                   ┆ Uprising                        │
│ 3            ┆ Coldplay               ┆ The Scientist                   │
│ 4            ┆ Five Iron Frenzy       ┆ Canada                          │
│ 5            ┆ Taylor Swift           ┆ You Belong With Me              │
│ 6            ┆ Cage The Elephant      ┆ Ain't No Rest For The Wicked (… │
│ 7            ┆ Kings Of Leon          ┆ Revelry                         │
│ 8            ┆